# Red de transferencias de tokens Ethereum

**Proyecto final — Estructura de Datos**
**Universidad Industrial de Santander**

**Profesor:** Jonnathan Alfredo Ramos Chaux
**Equipo:** Jhon Arciniegas, Pablo Álvarez y Emerson Ballesteros (líder)

---

## ¿De qué trata el proyecto?

Analizamos un **grafo** construido a partir de transferencias de un token
ERC-20 de Ethereum. Las direcciones son los **nodos** y las transferencias
son las **aristas** (dirigidas y con peso = cantidad transferida).

A partir de ese grafo resolvemos **4 tareas**, una por integrante:

| Tarea | Pregunta | Algoritmo | Responsable |
|-------|----------|-----------|-------------|
| 1 | ¿Cuál es el camino más barato entre dos direcciones? | Bellman-Ford | Pablo Álvarez |
| 2 | ¿Qué nodos son los más importantes? | Centralidad + PageRank | Jhon Arciniegas |
| 3 | ¿Qué grupos de nodos interactúan más entre sí? | Louvain (comunidades) | Emerson Ballesteros |
| 4 | ¿Cuál es la red mínima que conecta a todos? | Kruskal (MST) | Grupal |

**Reglas del proyecto:**
- No se permite usar el algoritmo de **Dijkstra**.
- Todo se implementa desde cero con Python (listas y diccionarios).


## Configuración

Ejecuta la siguiente celda para importar todo lo necesario. Solo usamos la
librería estándar de Python y `matplotlib` (para dibujar histogramas).


In [ ]:
# Librerías estándar
import csv
from collections import defaultdict

# Para graficar
import matplotlib.pyplot as plt

# Nuestros módulos (implementados desde cero)
from src.grafo import Grafo
from src.bellman_ford import bellman_ford, reconstruir_camino
from src.centralidad import centralidad_grado, pagerank, top_n
from src.comunidades import simetrizar, modularidad, louvain, resumen_comunidades
from src.arbol_expansion import UnionFind, aristas_no_dirigidas, kruskal, resumen_mst

# Rutas de los archivos de ejemplo
RUTA_TRANSFERENCIAS = "data/sample/transferencias.csv"
RUTA_ETIQUETAS = "data/sample/etiquetas.csv"

print("Todo listo.")


## Sección 1 — El dataset

Usamos un archivo CSV con transferencias de tokens. Cada fila es una
transferencia de una dirección a otra:

| Columna | Significado |
|---------|-------------|
| `from_address` | dirección que envía |
| `to_address` | dirección que recibe |
| `value` | cantidad transferida |

Para este ejemplo usamos una **muestra pequeña** (12 direcciones y 24
transferencias) para que todo se entienda y corra rápido. La misma idea se
aplica al dataset completo (millones de filas).


In [ ]:
# Cargamos el grafo desde el CSV.
grafo = Grafo()
grafo.cargar_transferencias(RUTA_TRANSFERENCIAS)
grafo.cargar_etiquetas(RUTA_ETIQUETAS)

print("Resumen del grafo:")
resumen = grafo.resumen()
for clave, valor in resumen.items():
    print(f"  {clave}: {valor}")

# Grado (número de conexiones) de cada nodo.
grados = {}
for u, vecinos in grafo.adj.items():
    grados[u] = grados.get(u, 0) + len(vecinos)   # aristas de salida
    for v, _ in vecinos:
        grados[v] = grados.get(v, 0) + 1          # aristas de entrada

# Histograma de grados.
plt.figure(figsize=(6, 4))
plt.hist(list(grados.values()), bins=range(0, 10), color="steelblue", edgecolor="white")
plt.xlabel("Grado")
plt.ylabel("Frecuencia")
plt.title("Distribución de grados")
plt.show()


## Tarea 1 — Caminos de costo mínimo (Bellman-Ford)

**Responsable:** Pablo Álvarez

**Problema:** queremos saber cuál es la ruta más barata (menor cantidad
transferida) para ir de una dirección a otra.

**Equivalencia matemática:** camino de costo mínimo en un grafo dirigido y
ponderado. El costo de un camino es la suma de los pesos de sus aristas.

**Alternativas consideradas:**

| Algoritmo | ¿Por qué sí / no? |
|-----------|-------------------|
| Dijkstra | Prohibido por las reglas del proyecto |
| BFS | Solo sirve cuando todas las aristas pesan lo mismo |
| **Bellman-Ford** | Funciona con pesos reales y detecta ciclos negativos |

**Elegimos Bellman-Ford.** Su complejidad es $O(|V| \cdot |E|)$, aceptable
para grafos de tamaño moderado.


In [ ]:
# Elegimos un origen con nombre conocido para que el resultado sea legible.
origen = "0x1111"   # Uniswap
distancias, previos = bellman_ford(grafo.adj, origen)

print(f"Caminos más baratos desde {grafo.nombre(origen)}:")
for destino in sorted(grafo.nodos):
    d = distancias[destino]
    if d == float("inf"):
        print(f"  {grafo.nombre(destino):<12} : inalcanzable")
    else:
        camino = reconstruir_camino(previos, origen, destino)
        nombres = " -> ".join(grafo.nombre(n) for n in camino)
        print(f"  {grafo.nombre(destino):<12} : costo {d:>7.1f}   ({nombres})")


## Tarea 2 — Centralidad e influencia

**Responsable:** Jhon Arciniegas

**Problema:** ¿cuáles son los nodos más importantes de la red?

**Equivalencia matemática:** dos medidas de centralidad:

- **Grado:** $C(v) = \text{grado}(v) / (|V| - 1)$
- **PageRank:** $PR(v) = \frac{1-d}{N} + d \sum_{u \to v} \frac{PR(u)}{\text{grado_salida}(u)}$

**Alternativas:** la centralidad de cercanía y la de intermediación también
miden importancia, pero requieren calcular caminos mínimos repetidamente. Para
un primer análisis bastan el grado y PageRank.


In [ ]:
# Centralidad de grado.
cent_grado = centralidad_grado(grafo.adj, grafo.nodos)

print("Top por centralidad de grado:")
for addr, score, nombre in top_n(cent_grado, n=5, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<12} : {score:.3f}")

# PageRank.
pr = pagerank(grafo.adj, grafo.nodos)
print("\nTop por PageRank:")
for addr, score, nombre in top_n(pr, n=5, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<12} : {score:.3f}")


## Tarea 3 — Detección de comunidades

**Responsable:** Emerson Ballesteros (líder)

**Problema:** ¿qué grupos de direcciones interactúan mucho entre sí?

**Equivalencia matemática:** maximizar la **modularidad** $Q$. Una partición
es buena si hay muchas aristas *dentro* de los grupos y pocas *entre* grupos.

**Alternativas consideradas:**

| Algoritmo | ¿Por qué sí / no? |
|-----------|-------------------|
| Girvan-Newman | Muy lento para grafos grandes |
| Clustering espectral | Necesita álgebra lineal densa |
| **Louvain** | Rápido y da buenos resultados |

**Elegimos Louvain.** Aquí implementamos su fase principal de forma sencilla:
movemos cada nodo a la comunidad vecina que más aumente $Q$.


In [ ]:
# Las comunidades se buscan sobre el grafo NO dirigido.
adj_no_dirigido = simetrizar(grafo.adj)

comunidades = louvain(adj_no_dirigido)
Q = modularidad(adj_no_dirigido, comunidades)

print(f"Modularidad Q = {Q:.3f}")
print(f"Número de comunidades = {len(set(comunidades.values()))}\n")

grupos = resumen_comunidades(comunidades, grafo.etiquetas)
for i, (c, nombres) in enumerate(sorted(grupos.items()), 1):
    print(f"Comunidad {i}: {', '.join(sorted(nombres))}")


## Tarea 4 — Árbol de expansión mínima (MST)

**Responsabilidad:** grupal

**Problema:** ¿cuál es el conjunto mínimo de conexiones que mantiene a toda
la red conectada?

**Equivalencia matemática:** árbol de expansión mínima. Es un árbol (sin
ciclos) que toca todos los nodos con el menor costo total.

**Alternativas consideradas:**

| Algoritmo | ¿Por qué sí / no? |
|-----------|-------------------|
| Prim | Necesita una cola de prioridad |
| **Kruskal** | Ordena aristas y usa Union-Find |

**Elegimos Kruskal** porque Union-Find es una estructura de datos clásica que
vale la pena estudiar.


In [ ]:
# Convertimos las aristas dirigidas en no dirigidas.
aristas_nd = aristas_no_dirigidas(grafo.aristas)

mst, costo = kruskal(grafo.nodos, aristas_nd)
print(f"MST: {len(mst)} aristas, costo total = {costo:.1f}\n")

print("Aristas del árbol de expansión mínima:")
for a, b, w in sorted(mst, key=lambda e: e[2]):
    print(f"  {grafo.nombre(a):<12} -- {grafo.nombre(b):<12}  (peso {w:.1f})")


## Conclusiones

- Un mismo dataset se puede atacar con **varios algoritmos de grafos**, y
  cada uno responde una pregunta distinta.
- **Bellman-Ford** permite caminos de costo mínimo sin usar Dijkstra.
- **PageRank** destaca a los nodos que concentran el flujo de la red.
- **Louvain** separa la red en comunidades coherentes (por ejemplo, el
  ecosistema de los DEX frente al de los exchanges).
- **Kruskal + Union-Find** muestra la "columna vertebral" mínima de la red.

Lo importante no es memorizar los algoritmos, sino entender **qué estructura
de datos usa cada uno** y **por qué funciona**.
